In [1]:
import os
import pandas as pd
from tqdm import tqdm
import numpy as np

import warnings
warnings.filterwarnings('ignore')

In [2]:
start_time = '2015-01-01'
end_time = '2023-04-01'

In [3]:
data_path = '/data/GRACE_data/A_share'

In [4]:
valid_stock_list = np.unique(pd.read_csv(f'{data_path}/stock_pool.csv', index_col = 'date').values.reshape(-1)).tolist()
valid_stock_list.sort()

In [5]:
def isnumber(x):
    try:
        float(x)
        return False
    except:
        return True

In [6]:
for stock_name in tqdm(valid_stock_list):
    kline_data = pd.read_csv(f'{data_path}/kline_day/{stock_name}.csv', index_col = 'date')
    basic_data = pd.read_csv(f'{data_path}/basic_factor//{stock_name}.csv', index_col = 'date')
    index_data = pd.read_csv(f'{data_path}/index_kline_day/000001.XSHG.csv', index_col = 'date')
    factor_loading_data = pd.read_csv(f'{data_path}/FF5_factor_loading/{stock_name}.csv', index_col = 'date')
    
    kline_data[['a_share_market_val_in_circulation','du_return_on_equity_ttm', 'inc_revenue_ttm','total_asset_turnover_ttm', 'debt_to_asset_ratio_ttm']] \
        = basic_data[['a_share_market_val_in_circulation','du_return_on_equity_ttm', 'inc_revenue_ttm','total_asset_turnover_ttm', 'debt_to_asset_ratio_ttm']]
    kline_data['log_ret'] = np.log(kline_data['close']/kline_data['close'].shift(1))
    kline_data['ret'] = kline_data['close']/kline_data['close'].shift(1) - 1
    kline_data['ret_5'] = kline_data['ret'].rolling(5).mean()
    kline_data['ret_10'] = kline_data['ret'].rolling(10).mean()
    kline_data['ret_20'] = kline_data['ret'].rolling(20).mean()
    kline_data['ret_30'] = kline_data['ret'].rolling(30).mean()

    kline_data.loc[(kline_data['volume'] < 1)|(kline_data['total_turnover'] < 1), :] = np.nan
    kline_data[(kline_data == np.inf) | (kline_data == -1 * np.inf)] = np.nan
    kline_data[kline_data.applymap(isnumber)] = np.nan
    kline_data['c_code'] = stock_name
    
    kline_data['index_ret'] = index_data['close']/index_data['close'].shift(1) - 1
    kline_data['excess_ret'] = kline_data['ret'] - kline_data['index_ret']
    
    kline_data[['Mkt-RF_loading', 'SMB_loading', 'HML_loading', 'RMW_loading', 'CMA_loading']] = \
        factor_loading_data[['Mkt-RF_loading', 'SMB_loading', 'HML_loading', 'RMW_loading', 'CMA_loading']]
        
    kline_data = kline_data.loc[start_time:end_time, :]
    kline_data.fillna(method = 'ffill', inplace = True)
    kline_data.fillna(0.0, inplace=True)
    
    kline_data.to_csv(f'{data_path}/daily_feature/{stock_name}.csv')

100%|██████████| 1682/1682 [03:18<00:00,  8.45it/s]


In [12]:
FF5_df = pd.read_csv(f'{data_path}/FF_5.csv')
ymd = pd.to_datetime(FF5_df['date'], format='%Y/%m/%d')
FF5_df['date'] = ymd.apply(lambda x: f'{x.year}-{x.month:02d}-{x.day:02d}')
FF5_df.set_index('date', inplace = True)
FF5_df.sort_index(inplace = True)
FF5_df['Mkt-RF'] = FF5_df['MKT'] - FF5_df['RF']

In [13]:
date_array = pd.read_csv(f'{data_path}/index_kline_day/000001.XSHG.csv')['date'].values

In [14]:
for factor_name in ['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA']:
    factor_df = pd.DataFrame(index = date_array)
    factor_df.index.name = 'date'
    factor_df['ret'] = FF5_df[factor_name]
    factor_df['ret_5'] = factor_df['ret'].rolling(5).mean()
    factor_df['ret_10'] = factor_df['ret'].rolling(10).mean()
    factor_df['ret_20'] = factor_df['ret'].rolling(20).mean()
    factor_df['ret_30'] = factor_df['ret'].rolling(30).mean()
    factor_df['close'] = 0.0
    
    factor_df[['Mkt-RF_loading', 'SMB_loading', 'HML_loading', 'RMW_loading', 'CMA_loading']] = 1
    
    factor_df = factor_df.loc[start_time:end_time, :]
    factor_df.fillna(method = 'ffill', inplace = True)
    factor_df.fillna(0.0, inplace=True)
    factor_df.to_csv(f'{data_path}/daily_feature/{factor_name}.csv')